In [ ]:
import requests, re, csv, os, glob, shutil, urllib3
import pandas as pd
import sqlite3
import calendar
from pathlib import Path
from datetime import datetime
from urllib.parse import unquote

# `手動取得`資料來源每月更新的最新單筆CSV

In [ ]:
# 將 11408 格式，轉換為 2025-08-31（對應月份的最後一天）
def change_date(adyear):
    original_data = adyear
    year, month = (1911 + int(original_data[:3]), int(original_data[3:]))
    # 取得該月最後一天
    last_day = calendar.monthrange(year, month)[1]
    date = datetime(year, month, last_day).strftime("%Y-%m-%d")

    return date

In [ ]:
# 取得當前工作目錄（路徑）
current_dir = Path.cwd()
print(f"目前工作目錄（路徑）是: {current_dir}")

In [ ]:
# 🔥 核心：這行可以封鎖 InsecureRequestWarning 警告
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

html_url = "https://data.gov.tw/dataset/168384"
responsts = requests.get(html_url, verify=False)
code = responsts.status_code
print(f"HTML狀態碼：{code}")

In [ ]:
urls = re.findall(r'"contentUrl":"(.*?)"', responsts.text)
print(urls)

In [ ]:
# 取得最後一筆
api_url = urls[-1]

In [ ]:
# 拼出 download 資料夾的路徑（記得加引號）
download_dir = current_dir / "download_entity"
# 如果該資料夾還不存在，就自動建立它
download_dir.mkdir(parents=True, exist_ok=True)

In [ ]:
response = requests.get(api_url, allow_redirects=True)
# 從 headers 中取得檔名
content_disposition = response.headers.get("Content-Disposition")
filename = re.search(r"filename\*\=UTF-8\'\'(.+)", content_disposition)
filename = unquote(filename.group(1))  # 解碼 URL 編碼的檔名
filename

In [ ]:
# 檔案儲存路徑（包含檔名.csv）
save_path = download_dir / f"{filename}"

In [ ]:
with open(save_path, "w", encoding="utf-8-sig") as file:
    file.write(response.text)

print(f"{filename}下載完成，路徑：{save_path}")

In [ ]:
# 取得擺放所有 csv檔案 的資料夾路徑
# folder_path = download_dir
folder_path = current_dir / "download_entity"

# 錯誤CSV檔案的資料夾路徑（記得加引號）
mistake_folder = current_dir / "download_mistake"
# 如果該資料夾還不存在，就自動建立它
mistake_folder.mkdir(parents=True, exist_ok=True)

# 使用 pathlib 的 .glob 找出所有 CSV 檔案
csv_files = list(folder_path.glob("*.csv"))  # 或直接使用 download_dir 即可

In [ ]:
# for 一次處理所有檔案
count = 0
for file in csv_files:
    df = pd.read_csv(file)

    if len(df) != 26 and df.iloc[0, 1] == "計":
        # .stem取檔名／.suffix取副檔名
        name = file.stem  # 取得不含副檔名的檔名
        # print(f"原完整檔名: {file.name}, 主檔名: {name}, 副檔名: {file.suffix}")
        print(f"發現錯誤檔案!! 原完整檔名: {file.name}")

        # 將「原始錯誤檔案」複製一份到 mistake_folder 備份，file 是 pathlib.Path 物件，shutil.copy 會完美處理它
        shutil.copy(file, mistake_folder / file.name)
        print(f"-> 已成功備份錯誤檔案至: {mistake_folder / file.name}")

        # 取得所有 不是「計」 的列資料
        df = df[df["性別"] != "計"]

        # 【關鍵修改】儲存路徑直接指向 file (原始檔案路徑)
        save_path = file

        df.to_csv(
            save_path,
            encoding="utf-8-sig",
            index=False,
        )
        print(f"-> {file.name} 原始檔案已成功修正，並覆蓋！\n")
        count += 1
print(f"共修改並覆蓋了 {count} 個檔案")

In [ ]:
# for 一次處理所有檔案
count = 0
for file in csv_files:
    df = pd.read_csv(file)

    # 全object物件/string字串欄位去除前後空格
    text_cols = df.select_dtypes(include=["object", "string"]).columns
    # 只對這些文字欄位做 strip()
    df[text_cols] = df[text_cols].apply(lambda col: col.str.strip())

    # 將要判斷的那個格子轉為數字型態，並清掉可能殘留的逗號
    check_value = pd.to_numeric(str(df.iloc[0, 2]).replace(",", ""), errors="coerce")

    # 用安全轉換後的 check_value 進行判斷
    # pd.isna(check_value) 可以順便防止那個格子不小心讀到文字或空值
    if (not pd.isna(check_value)) and (
        check_value < 200000 or isinstance(check_value, (int, float))
    ):
        # .stem取檔名／.suffix取副檔名
        name = file.stem  # 取得不含副檔名的檔名
        # print(f"原完整檔名: {file.name}, 主檔名: {name}, 副檔名: {file.suffix}")
        print(f"發現錯誤檔案!! 原完整檔名: {file.name}")

        # 將「原始錯誤檔案」複製一份到 mistake_folder 備份，file 是 pathlib.Path 物件，shutil.copy 會完美處理它
        shutil.copy(file, mistake_folder / file.name)
        print(f"-> 已成功備份錯誤檔案至: {mistake_folder / file.name}")

        # 一次清除千分位逗號與將不同形式的「-」轉為 0
        df = df.replace({",": "", "-": 0}, regex=True)

        # 確保被修改的欄位（第 2 欄以後）都先變成數字，避免轉 int 失敗
        df.iloc[:, 2:] = df.iloc[:, 2:].apply(pd.to_numeric, errors="coerce").fillna(0)
        # 值向右移動並轉成整數
        df.iloc[:, 2:] = df.iloc[:, 2:].shift(1, axis=1).fillna(0).astype(int)

        # 【關鍵修改】儲存路徑直接指向 file (原始檔案路徑)
        save_path = file

        df.to_csv(
            save_path,
            encoding="utf-8-sig",
            index=False,
        )
        print(f"-> {file.name} 原始檔案已成功修正，並覆蓋！\n")
        count += 1
print(f"共修改並覆蓋了 {count} 個檔案")

In [ ]:
# 整理[年齡]欄標題名稱 中文 → 英文
cols = ["region", "gender"] + [f"{i}aged" for i in range(0, 101)]

In [ ]:
df_list = []
for file in csv_files:
    df = pd.read_csv(file)

    # 刪除欄位「總計」
    if "總計" in df.columns:
        df.drop(["總計"], axis=1, inplace=True)

    # 整理欄標題名稱
    df.columns = cols

    # 調整gender欄位裡 value 問題 → ['男', '女', '女 ', 1, 2]
    df["gender"] = df["gender"].replace({"女 ": "女", 1: "男", 2: "女"})

    # 調整Region欄位裡 value 問題 → ['八德市', '大溪鎮', '蘆竹鄉',]
    df["region"] = df["region"].str.replace(r"(市|鎮|鄉)$", "區", regex=True)

    # 先重置索引，確保沒有重複的 index 搗亂
    df = df.reset_index(drop=True)

    # 取得第 3 欄之後的所有欄位名稱
    age_cols = df.columns[3:]

    # 先轉字串，每一欄都做 strip() 徹底解決「前後帶有空格」的隱形殺手，並拿掉逗號
    numeric_block = (
        df[age_cols]
        .astype(str)
        .apply(lambda col: col.str.strip().str.replace(",", "", regex=False))
    )

    # 強制轉成數字，無法轉換的怪文字會變成 NaN
    numeric_block = numeric_block.apply(pd.to_numeric, errors="coerce")

    # 補 0 並轉成純整數型態 (int64)
    numeric_block = numeric_block.fillna(0).astype("int64")

    # 【關鍵】直接把乾淨的整數 DataFrame 覆蓋回去
    df[age_cols] = numeric_block

    # 【在這裡加上 .copy() 重新整理記憶體】
    df = df.copy()
    # 取得檔名裡的民國年月，新增欄位 'date' 西元格式，並指定位置 loc= 0
    ymd = file.stem[:5]
    data_ymd = change_date(ymd)

    # 🔥建立一個只有 date 的乾淨 DataFrame
    date_df = pd.DataFrame({"date": [data_ymd] * len(df)}, index=df.index)

    # 🔥用 concat 一口氣橫向拼接（axis=1），這會自動生成一個最有效率、完全沒有碎片的 DataFrame
    df = pd.concat([date_df, df], axis=1)

    df_list.append(df)

In [ ]:
# 合併所有 CSV檔案，並重新編排索引，避免重複。
dfmerge = pd.concat(df_list, ignore_index=True)

In [ ]:
combined_df = dfmerge.copy()

In [ ]:
# 檢查 gender 值
combined_df["gender"].unique()

In [ ]:
# 檢查 Region 值
combined_df["region"].unique()

In [ ]:
# 檢查 values 裡的型態type() 是否一致 [全部欄位／單一欄位]

# [全部欄位] 使用 .values.flatten() 把整個 0~100aged 區塊拉平，直接用 Python 的 set 抓出所有型態
unique_types = set(type(val) for val in combined_df.iloc[:, 3::].values.flatten())
print("[全部欄位] 0~100aged 欄位的value型態：", unique_types)
# # 合併所有欄位的 set，取出唯一類型
# unique_types = set().union(*column_types)
# print("0~100aged 欄位的value型態：", unique_types)

print()
# [單一欄位]檢查 單一欄位中所有值的類型，列出該欄位所有值的類型集
print(
    "[單一欄位] 0aged 欄位的value型態：", set(type(val) for val in combined_df["0aged"])
)

In [ ]:
# 建立 SQLite 連線（或連到現有資料庫）
conn = sqlite3.connect("taoyuanage.db")

# if_exists='replace' ←← 若有資料全部覆寫 ／ index=False ←← 不要將index的0,1,2,3...寫入資料庫
dfmerge.to_sql("dfmerge", conn, if_exists="replace", index=False)

# 關開連接
conn.close()

In [ ]:
# 建立 SQLite 連線（或連到現有資料庫）
conn = sqlite3.connect("taoyuanage.db")

# pd.read_sql() 的第一個參數必須是『SQL 查詢語句』，第二個參數必須是 conn ←開啟連接資料庫
datas = pd.read_sql("SELECT * FROM dfmerge", conn)

# 關閉連線
conn.close()

In [ ]:
# 顯示前 N 筆資料，不填數字為預設值 5筆
df = pd.DataFrame(datas)
df.head()

In [ ]:
df.columns

In [ ]:
df.info()